# Notebook 18: Stock Market Prediction with LSTM
**DCS 404 — Data Science and Machine Learning**

---

## Learning Objectives
- Understand time series forecasting and its unique challenges
- Prepare a sliding-window dataset for sequence prediction
- Build and train an LSTM for one-step-ahead forecasting
- Evaluate with MAE, RMSE, and directional accuracy
- Critically assess the limits of ML-based stock prediction

> **Disclaimer**: This notebook is purely educational. No investment advice is implied.

## 1. Time Series vs Standard ML

| | Standard ML | Time Series |
|--|-------------|-------------|
| Sample order | Irrelevant | Critical |
| Train/test split | Random shuffle ok | Chronological only |
| Features | Given | Often lag-engineered |
| Target | Fixed label | Future values |

**The sliding window**: convert a sequence into supervised learning examples:
```
Sequence: [p1, p2, p3, p4, p5, p6, ...]
Window=3:   X=[p1,p2,p3] → y=p4
            X=[p2,p3,p4] → y=p5
            X=[p3,p4,p5] → y=p6
```

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings; warnings.filterwarnings('ignore')

torch.manual_seed(42); np.random.seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

## 2. Synthetic Stock Data (Geometric Brownian Motion)

$$S_t = S_{t-1} \cdot \exp\left[(\mu - \tfrac{\sigma^2}{2})\Delta t + \sigma\sqrt{\Delta t}\,\epsilon_t\right], \quad \epsilon_t \sim \mathcal{N}(0,1)$$

This is the continuous-time model underpinning Black-Scholes option pricing. It produces realistic-looking price series.

In [ ]:
def gbm(S0=100, mu=0.0003, sigma=0.018, T=1200):
    prices = [S0]
    for _ in range(T-1):
        prices.append(prices[-1] * np.exp((mu - 0.5*sigma**2) + sigma*np.random.randn()))
    return np.array(prices)

prices = gbm()
dates = pd.date_range('2019-01-01', periods=1200, freq='B')
df = pd.DataFrame({'Price': prices}, index=dates)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(df.index, df['Price'], color='steelblue')
ax.set_title('Simulated Stock Price (GBM)')
ax.set_ylabel('Price ($)')
plt.tight_layout(); plt.show()

## 3. Feature Engineering

In [ ]:
df['MA10']   = df['Price'].rolling(10).mean()
df['MA30']   = df['Price'].rolling(30).mean()
df['Std10']  = df['Price'].rolling(10).std()
df['Return'] = df['Price'].pct_change()
df['High5']  = df['Price'].rolling(5).max()
df['Low5']   = df['Price'].rolling(5).min()
df = df.dropna()

fig, axes = plt.subplots(2, 1, figsize=(12, 6))
axes[0].plot(df.index, df['Price'],  label='Price')
axes[0].plot(df.index, df['MA10'],   label='MA10',  linestyle='--')
axes[0].plot(df.index, df['MA30'],   label='MA30',  linestyle=':')
axes[0].set_title('Price with Moving Averages'); axes[0].legend()

axes[1].bar(df.index, df['Return'], alpha=0.6, color='tomato', width=1)
axes[1].axhline(0, color='black', linewidth=0.5)
axes[1].set_title('Daily Returns')
plt.tight_layout(); plt.show()

## 4. Building the Dataset

In [ ]:
FEAT_COLS = ['Price','MA10','MA30','Std10','Return','High5','Low5']

# Scale all features to [0,1]
feat_scaler  = MinMaxScaler()
price_scaler = MinMaxScaler()
scaled     = feat_scaler.fit_transform(df[FEAT_COLS].values.astype(np.float32))
price_sc   = price_scaler.fit_transform(df[['Price']].values.astype(np.float32))

LOOKBACK = 30

def make_windows(features, target, lookback):
    X, y = [], []
    for i in range(len(features) - lookback):
        X.append(features[i:i+lookback])
        y.append(target[i+lookback])
    return np.array(X, np.float32), np.array(y, np.float32)

X_all, y_all = make_windows(scaled, price_sc, LOOKBACK)

# Chronological split — NEVER shuffle!
split = int(0.8 * len(X_all))
X_tr, X_va = X_all[:split], X_all[split:]
y_tr, y_va = y_all[:split], y_all[split:]
print(f'Train: {X_tr.shape}, Val: {X_va.shape}')

## 5. LSTM Model

In [ ]:
class StockLSTM(nn.Module):
    def __init__(self, n_features, hidden=64, layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(n_features, hidden, layers,
                            batch_first=True, dropout=dropout)
        self.fc = nn.Sequential(
            nn.Linear(hidden, 32), nn.ReLU(), nn.Linear(32, 1)
        )
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

model = StockLSTM(len(FEAT_COLS)).to(device)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
X_tr_t = torch.tensor(X_tr).to(device); y_tr_t = torch.tensor(y_tr).to(device)
X_va_t = torch.tensor(X_va).to(device); y_va_t = torch.tensor(y_va).to(device)

train_dl = DataLoader(TensorDataset(X_tr_t, y_tr_t), batch_size=32, shuffle=True)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=5e-4, weight_decay=1e-5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=10, factor=0.5)

train_losses, val_losses = [], []
best_val = float('inf')

for epoch in range(80):
    model.train()
    ep_loss = 0
    for Xb, yb in train_dl:
        optimizer.zero_grad()
        loss = criterion(model(Xb), yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        ep_loss += loss.item()
    tl = ep_loss / len(train_dl)

    model.eval()
    with torch.no_grad():
        vl = criterion(model(X_va_t), y_va_t).item()
    scheduler.step(vl)
    train_losses.append(tl); val_losses.append(vl)

    if vl < best_val:
        best_val = vl
        torch.save(model.state_dict(), 'best_lstm.pt')

    if (epoch+1) % 20 == 0:
        print(f'Epoch {epoch+1:3d} | Train={tl:.5f} | Val={vl:.5f}')

In [ ]:
model.load_state_dict(torch.load('best_lstm.pt', map_location=device))
model.eval()

with torch.no_grad():
    y_pred_sc = model(X_va_t).cpu().numpy()

y_pred = price_scaler.inverse_transform(y_pred_sc)
y_true = price_scaler.inverse_transform(y_va)

mae  = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
print(f'MAE={mae:.2f}   RMSE={rmse:.2f}   MAPE={mape:.2f}%')

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 8))
axes[0].plot(train_losses, label='Train'); axes[0].plot(val_losses, label='Val')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('MSE'); axes[0].set_title('Loss Curve')
axes[0].legend()

axes[1].plot(y_true,  label='Actual',     linewidth=1.5)
axes[1].plot(y_pred,  label='Predicted',  linewidth=1.5, linestyle='--')
axes[1].fill_between(range(len(y_true)),
                      y_pred.ravel()-rmse, y_pred.ravel()+rmse,
                      alpha=0.15, color='orange', label='±RMSE')
axes[1].set_xlabel('Time Step'); axes[1].set_ylabel('Price ($)')
axes[1].set_title(f'LSTM Forecast  |  MAE=${mae:.2f}  RMSE=${rmse:.2f}  MAPE={mape:.1f}%')
axes[1].legend()
plt.tight_layout(); plt.show()

## 6. Directional Accuracy

In [ ]:
dir_pred = np.sign(y_pred[1:] - y_pred[:-1])
dir_true = np.sign(y_true[1:] - y_true[:-1])
da = (dir_pred == dir_true).mean()
print(f'Directional accuracy: {da:.2%}')
print('(A random guesser achieves 50%)')

## 7. Critical Assessment

### What This Model Does Well
- Captures short-term momentum from moving averages
- Learns smooth trends in normalised data

### What It Cannot Do
- Predict reversals caused by breaking news
- Generalise to market regime changes
- Beat a buy-and-hold strategy consistently

### The Efficient Market Hypothesis
In a semi-strong efficient market, prices already reflect all publicly available information. If a simple LSTM could reliably profit, everyone would use it and the signal would disappear immediately.

**Lesson**: low MAPE on historical data does not mean the model will be profitable on live trading.

## Exercises

1. Try `LOOKBACK = [10, 20, 30, 60]`. Which gives the lowest validation RMSE?
2. Add RSI (Relative Strength Index) as a feature. Does it improve accuracy?
3. Compare GRU vs LSTM: parameters, training time, RMSE.
4. Download real data with `yfinance` and apply the same pipeline to AAPL or BTC-USD.
5. Build a simple trading strategy: buy when the model predicts a price increase. Calculate total return vs buy-and-hold.

## Reflection Questions
- Why must we never shuffle time series data before splitting?
- A model trained on 2010–2020 data is deployed in 2020. What problems might arise?
- If MAPE = 1% sounds great, why might this model still be useless for trading?

---

## Course Summary

Congratulations on completing DCS 404!

| Module | Topic |
|--------|-------|
| 01 | AI fundamentals: search, A*, Minimax |
| 02-04 | NumPy, Pandas, Matplotlib |
| 05 | Data preparation and feature engineering |
| 06 | Linear regression and regularisation |
| 07-11 | Classification: logistic, SVM, KNN, NB, trees, ensembles |
| 12 | Text classification (20 Newsgroups) |
| 13 | Model evaluation and selection |
| 14 | Clustering: K-Means, hierarchical, DBSCAN |
| 15-17 | Deep learning: foundations, PyTorch, CNN, LSTM |
| 18 | Applied LSTM: stock price forecasting |

**What next?**: Transformers & Attention, Reinforcement Learning, Graph Neural Networks, Generative Models (VAE, GAN, Diffusion Models)